# Import thư viện

In [1]:
import pandas as pd
import numpy as np

# Load data

In [2]:
df = pd.read_csv("../../Dataset/data_cleaned/country_vaccinations_added_fields.csv", index_col=0,
                 parse_dates=['date'])

df.head(10)

,iso_code,country,date,total_vaccinations,people_vaccinated,people_fully_vaccinated,daily_vaccinations_raw,daily_vaccinations,total_vaccinations_per_hundred,people_vaccinated_per_hundred,...,vaccine_id,vaccine_names,source_id,source_name,source_website,dropout_date,vaccine_count,coverage_group,daily_vaccination_7d_avg,vaccination_momentum
0,AFG,Afghanistan,2021-02-22,0,0,0,0,0,0.00,0.00,...,"V002, V008, V009","BBIBP-CorV, Oxford/AstraZeneca, Pfizer/BioNTech",S001,World Health Organization,https://covid19.who.int/,0.0,3,Low,0.000000,0.000000
1,AFG,Afghanistan,2021-02-23,0,0,0,0,1367,0.00,0.00,...,"V002, V008, V009","BBIBP-CorV, Oxford/AstraZeneca, Pfizer/BioNTech",S001,World Health Organization,https://covid19.who.int/,0.0,3,Low,683.500000,0.000000
2,AFG,Afghanistan,2021-02-24,0,0,0,0,1367,0.00,0.00,...,"V002, V008, V009","BBIBP-CorV, Oxford/AstraZeneca, Pfizer/BioNTech",S001,World Health Organization,https://covid19.who.int/,0.0,3,Low,911.333333,0.000000
3,AFG,Afghanistan,2021-02-25,0,0,0,0,1367,0.00,0.00,...,"V002, V008, V009","BBIBP-CorV, Oxford/AstraZeneca, Pfizer/BioNTech",S001,World Health Organization,https://covid19.who.int/,0.0,3,Low,1025.250000,0.000000
4,AFG,Afghanistan,2021-02-26,0,0,0,0,1367,0.00,0.00,...,"V002, V008, V009","BBIBP-CorV, Oxford/AstraZeneca, Pfizer/BioNTech",S001,World Health Organization,https://covid19.who.int/,0.0,3,Low,1093.600000,0.000000
5,AFG,Afghanistan,2021-02-27,0,0,0,0,1367,0.00,0.00,...,"V002, V008, V009","BBIBP-CorV, Oxford/AstraZeneca, Pfizer/BioNTech",S001,World Health Organization,https://covid19.who.int/,0.0,3,Low,1139.166667,0.000000
6,AFG,Afghanistan,2021-02-28,8200,8200,0,0,1367,0.02,0.02,...,"V002, V008, V009","BBIBP-CorV, Oxford/AstraZeneca, Pfizer/BioNTech",S001,World Health Organization,https://covid19.who.int/,100.0,3,Low,1171.714286,16.670732
7,AFG,Afghanistan,2021-03-01,0,0,0,0,1580,0.00,0.00,...,"V002, V008, V009","BBIBP-CorV, Oxford/AstraZeneca, Pfizer/BioNTech",S001,World Health Organization,https://covid19.who.int/,0.0,3,Low,1397.428571,0.000000
8,AFG,Afghanistan,2021-03-10,0,0,0,0,2862,0.00,0.00,...,"V002, V008, V009","BBIBP-CorV, Oxford/AstraZeneca, Pfizer/BioNTech",S001,World Health Organization,https://covid19.who.int/,0.0,3,Low,1611.000000,0.000000
9,AFG,Afghanistan,2021-03-11,0,0,0,0,2862,0.00,0.00,...,"V002, V008, V009","BBIBP-CorV, Oxford/AstraZeneca, Pfizer/BioNTech",S001,World Health Organization,https://covid19.who.int/,0.0,3,Low,1824.571429,0.000000


# Chia tập train/test

In [58]:
df = df.sort_values("date").reset_index(drop=True)

split=int(len(df)*0.8)

train = df.iloc[:split].copy()
test = df.iloc[split:].copy()

split_date = train["date"].max()
test = test[test["date"] > split_date].copy()

print("Tập Train", len(train))
print("Tập Test", len(test))

print("Train date", train['date'].min(),"->", train["date"].max())
print("Test date", test['date'].min(),"->", test["date"].max())

train.to_csv("../../Dataset/data_split/train.csv", index=False)
test.to_csv("../../Dataset/data_split/test.csv", index=False)

Tập Train 20689
Tập Test 4981
Train date 2020-12-02 00:00:00 -> 2021-05-24 00:00:00
Test date 2021-05-25 00:00:00 -> 2021-06-20 00:00:00


# Mô hình hồi quy tuyến tính

In [59]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

target = "people_vaccinated_per_hundred"

df[target] = df.groupby("country")[target].cummax()

start_date = train["date"].min()
train["days"] = (train["date"] - start_date).dt.days
test["days"] = (test["date"] - start_date).dt.days


X_train = pd.get_dummies(train[["days", "country"]], columns=["country"])
X_test = pd.get_dummies(test[["days", "country"]], columns=["country"])
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

y_train = train[target]
y_test = test[target]

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE  = {mae:.4f}")
print(f"RMSE = {rmse:.4f}")
print(f"R2   = {r2:.4f}")


MAE  = 8.8840
RMSE = 10.8652
R2   = 0.7621


In [53]:
y_pred = model.predict(X_test)

result = test[["country", "date", target]].copy()
result['predicted'] = y_pred

result['error']=result[target]-result['predicted']
result['abs error'] = result['error'].abs()

result.to_csv("../../Dataset/data_split/linear_regression_result.csv", index=False)

result.head()

,country,date,people_vaccinated_per_hundred,predicted,error,abs error
20881,Slovenia,2021-05-25,30.5300,26.8984,3.6316,3.6316
20882,Maldives,2021-05-25,56.9700,50.0193,6.9507,6.9507
20883,Somalia,2021-05-25,0.7900,5.1453,-4.3553,4.3553
20884,Ghana,2021-05-25,2.7300,11.4594,-8.7294,8.7294
20885,Saint Kitts and Nevis,2021-05-25,29.3600,25.8556,3.5044,3.5044
